# 실습 16. 불멸시간 편향과 시간의존 Cox 모형

사회약학 연구방법 노트의 '실습 16. 불멸시간 편향과 시간의존 Cox 모형'에 딸린 노트북입니다. 급성 심근경색 퇴원 환자 6,000명의 가상 자료에서, 효과가 없는 약물 X가 잘못된 분석에서는 위험비 0.37로 나오는 것(불멸시간 편향)을 직접 만들어 보고, 랜드마크 분석과 시간의존 Cox 모형으로 바로잡습니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 16에 있습니다.


## 가. 실습 데이터 준비

### 셀 1. lifelines 설치

In [ ]:
!pip install lifelines

### 셀 2. 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
d = pd.read_csv(BASE + "ami_cohort.csv", storage_options=UA)
print(d.shape)
d.head(5)

### 셀 3. 자료 확인

In [ ]:
print("환자", len(d), "명, id 중복", d["id"].duplicated().sum(), "건")
print("사망", d["died"].sum(), "명",
      f"({d['died'].mean() * 100:.1f}%)")
print("약물 X를 시작한 사람", d["rx_day"].notna().sum(), "명")
alive = d[d["died"] == 0]
print("생존자의 추적 일수:", alive["fu_days"].unique())
print("사망자의 추적 일수 최대:", d.loc[d["died"] == 1, "fu_days"].max())
bad = (d["rx_day"] >= d["fu_days"]).sum()
print("추적이 끝난 뒤의 처방", bad, "건")

### 셀 4. 약물 X를 시작한 시점의 분포

In [ ]:
rx_month = d["rx_day"] / 365 * 12          # 일 -> 개월
print(rx_month.describe().round(1))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(d["rx_day"].dropna(), bins=24)
ax.set_xlabel("Days from discharge to first prescription")
ax.set_ylabel("Number of patients")
plt.tight_layout()

## 나. 불멸시간 편향 재현하기

일부러 잘못된 분석을 합니다. 추적 중 한 번이라도 약을 시작한 사람을 처음부터 사용군으로 두고 퇴원일부터 추적합니다.

### 셀 5. 추적 중 사용 여부로 군 나누기 (잘못된 분류)

In [ ]:
d["ever"] = d["rx_day"].notna().astype(int)   # 1년 안에 시작 = 1

t1 = d.groupby("ever")["died"].agg(["count", "sum", "mean"])
t1.columns = ["n", "deaths", "risk"]
print("상대위험도:", round(t1.loc[1, "risk"] / t1.loc[0, "risk"], 2))
t1["risk"] = (t1["risk"] * 100).round(1)       # 1년 사망 위험(%)
t1

### 셀 6. 잘못된 분류로 그린 Kaplan-Meier 곡선

In [ ]:
from lifelines import KaplanMeierFitter

fig, ax = plt.subplots(figsize=(6.5, 4))
for g, name in [(1, "Ever users"), (0, "Never users")]:
    s = d[d["ever"] == g]
    km = KaplanMeierFitter().fit(s["fu_days"], s["died"], label=name)
    km.plot_survival_function(ax=ax, ci_show=False)
    print(name, "90일 생존율", round(km.predict(90), 3),
          " 365일 생존율", round(km.predict(365), 3))
ax.set_xlabel("Days since discharge")
ax.set_ylabel("Survival probability")
ax.set_ylim(0.6, 1)
plt.tight_layout()

### 셀 7. 잘못된 분류의 Cox 모형

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()
cph.fit(d, duration_col="fu_days", event_col="died", formula="ever")
cph.print_summary(decimals=3)

def hr_ci(model, name):
    r = model.summary.loc[name]
    lo, hi = r["exp(coef) lower 95%"], r["exp(coef) upper 95%"]
    return f"{r['exp(coef)']:.2f} ({lo:.2f}-{hi:.2f})"

print("HR (95% CI):", hr_ci(cph, "ever"))

셀 8은 일부러 오류를 내 보는 셀이어서 이 노트북에서는 뺐습니다(사이트의 실습 16 나 절에 있습니다). 셀 번호는 사이트와 같게 두었으므로 셀 7 다음이 셀 9입니다.

### 셀 9. 사용군의 불멸 인년 계산

In [ ]:
u = d[d["ever"] == 1]                  # 사용군 2,734명
nu = d[d["ever"] == 0]                 # 비사용군 3,266명
py_u = u["fu_days"].sum() / 365        # 사용군 인년 (퇴원일부터)
py_nu = nu["fu_days"].sum() / 365
immortal = u["rx_day"].sum() / 365     # 퇴원일부터 첫 처방일까지
print(f"사용군 {py_u:.0f}인년, 그중 첫 처방 전 {immortal:.0f}인년"
      f" ({immortal / py_u * 100:.0f}%)")
print(f"비사용군 {py_nu:.0f}인년")
rate_u = u["died"].sum() / py_u * 100       # 100인년당 사망률
rate_nu = nu["died"].sum() / py_nu * 100
print(f"사망률 {rate_u:.1f} 대 {rate_nu:.1f},"
      f" 발생률비 {rate_u / rate_nu:.2f}")

### 셀 10. 일찍 사망한 사람은 어느 군에 있는가

In [ ]:
print("사용군에서 첫 처방 전에 사망한 사람:",
      (u["fu_days"] < u["rx_day"]).sum(), "명")
early = d[(d["died"] == 1) & (d["fu_days"] <= 365 * 3 / 12)]
print("퇴원 후 3개월 안의 사망:", len(early), "건")
print(early["ever"].value_counts())

## 다. 랜드마크 분석

### 셀 11. 3개월 랜드마크 자료 만들기

In [ ]:
L = 365 * 3 / 12                           # 3개월 = 91.25일
lm = d[d["fu_days"] > L].copy()            # 그때 살아 있는 사람만
lm["exposed"] = (lm["rx_day"] <= L).astype(int)   # 그때까지 시작 = 1
lm["t_lm"] = lm["fu_days"] - L             # 랜드마크부터의 일수
print("대상자", len(lm), "명, 제외", len(d) - len(lm), "명")
late = ((lm["exposed"] == 0) & (lm["ever"] == 1)).sum()
print("비노출군 가운데 3개월 뒤에 시작한 사람", late, "명")

t2 = lm.groupby("exposed")["died"].agg(["count", "sum", "mean"])
t2.columns = ["n", "deaths", "risk"]
t2["risk"] = (t2["risk"] * 100).round(1)
t2

### 셀 12. 랜드마크부터의 Kaplan-Meier 곡선과 Cox 모형

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for g, name in [(1, "Exposed by 3 months"), (0, "Unexposed")]:
    s = lm[lm["exposed"] == g]
    km = KaplanMeierFitter().fit(s["t_lm"], s["died"], label=name)
    km.plot_survival_function(ax=ax, ci_show=False)
ax.set_xlabel("Days since the 3-month landmark")
ax.set_ylabel("Survival probability")
ax.set_ylim(0.6, 1)
plt.tight_layout()

cph_lm = CoxPHFitter().fit(lm, "t_lm", "died", formula="exposed")
print("3개월 랜드마크 HR (95% CI):", hr_ci(cph_lm, "exposed"))

### 셀 13. 랜드마크 시점을 1, 3, 6개월로 바꿔 보기

In [ ]:
def landmark(d, months):
    L = 365 * months / 12
    lm = d[d["fu_days"] > L].copy()
    lm["exposed"] = (lm["rx_day"] <= L).astype(int)
    lm["t_lm"] = lm["fu_days"] - L
    m = CoxPHFitter().fit(lm, "t_lm", "died", formula="exposed")
    e, ne = lm[lm["exposed"] == 1], lm[lm["exposed"] == 0]
    return {"month": months, "n": len(lm),
            "excluded": len(d) - len(lm), "exposed": len(e),
            "late start %": round(ne["ever"].mean() * 100, 1),
            "deaths": f"{e['died'].sum()} / {ne['died'].sum()}",
            "HR (95% CI)": hr_ci(m, "exposed")}

pd.DataFrame([landmark(d, m) for m in [1, 3, 6]])

## 라. 시간의존 Cox 모형

### 셀 14. 한 사람 한 줄 자료를 긴 형식으로 바꾸기

In [ ]:
u = d[d["ever"] == 1]        # 약을 시작한 사람: 두 줄
nu = d[d["ever"] == 0]       # 시작하지 않은 사람: 한 줄
before = pd.DataFrame({"id": u["id"], "start": 0.0,
                       "stop": u["rx_day"], "drugx": 0, "died": 0})
after = pd.DataFrame({"id": u["id"], "start": u["rx_day"],
                      "stop": u["fu_days"], "drugx": 1,
                      "died": u["died"]})
never = pd.DataFrame({"id": nu["id"], "start": 0.0,
                      "stop": nu["fu_days"], "drugx": 0,
                      "died": nu["died"]})
long = pd.concat([before, after, never])
long = long.sort_values(["id", "start"]).reset_index(drop=True)
print(len(d), "명 ->", len(long), "줄")
long.head(6)

### 셀 15. 표 16-4의 환자 세 명 확인

In [ ]:
ex = {298: "A", 64: "B", 3: "C"}       # 16장 표 16-4의 환자
t = long[long["id"].isin(ex)].copy()
t["patient"] = t["id"].map(ex)
t[["start", "stop"]] = t[["start", "stop"]].round().astype(int)
t = t.sort_values(["patient", "start"])
t[["patient", "id", "start", "stop", "drugx", "died"]]

### 셀 16. 긴 형식 자료 점검

In [ ]:
g = long.groupby("id")
prev_stop = g["stop"].shift()           # 같은 환자의 앞 줄 stop
second = long[prev_stop.notna()]        # 둘째 줄만
print("둘째 줄", len(second), "줄, 앞 줄의 끝에서 시작하는가:",
      (second["start"] == prev_stop.dropna()).all())
print("길이가 0 이하인 구간", (long["stop"] <= long["start"]).sum(), "줄")
last = g.tail(1)                        # 환자마다 마지막 줄
print("사망", long["died"].sum(), "건, 마지막 줄의 사망",
      last["died"].sum(), "건, 원자료", d["died"].sum(), "건")
py_long = (long["stop"] - long["start"]).sum() / 365
print(f"인년 합계 {py_long:.0f}, 원자료 {d['fu_days'].sum() / 365:.0f}")

### 셀 17. 노출 상태별 사망, 인년, 사망률

In [ ]:
long["py"] = (long["stop"] - long["start"]) / 365
t3 = long.groupby("drugx").agg(deaths=("died", "sum"),
                               py=("py", "sum"))
t3["rate"] = t3["deaths"] / t3["py"] * 100      # 100인년당 사망률
print("발생률비:", round(t3.loc[1, "rate"] / t3.loc[0, "rate"], 2))
t3.round({"py": 0, "rate": 1})

### 셀 18. 시간의존 Cox 모형

In [ ]:
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(long, id_col="id", event_col="died",
        start_col="start", stop_col="stop", formula="drugx")
ctv.print_summary(decimals=3)

### 셀 19. lifelines 함수로 같은 자료 만들기

In [ ]:
from lifelines.utils import to_long_format, add_covariate_to_timeline

base = to_long_format(d[["id", "fu_days", "died"]],
                      duration_col="fu_days")
rx = d.loc[d["ever"] == 1, ["id", "rx_day"]].assign(drugx=1)
long2 = add_covariate_to_timeline(base, rx, id_col="id",
                                  duration_col="rx_day",
                                  event_col="died")
long2["drugx"] = long2["drugx"].fillna(0).astype(int)
ctv2 = CoxTimeVaryingFitter().fit(long2, id_col="id",
    event_col="died", start_col="start", stop_col="stop")
print(len(long2), "줄, HR (95% CI):", hr_ci(ctv2, "drugx"))
long2.head(4)

### 셀 20. 세 분석의 위험비 비교

In [ ]:
pd.DataFrame({"HR (95% CI)": {
    "ever vs never use, from discharge": hr_ci(cph, "ever"),
    "landmark at 3 months": hr_ci(cph_lm, "exposed"),
    "time-varying exposure": hr_ci(ctv, "drugx")}})

## 과제 정답

**과제 1.** 3개월 랜드마크 분석에서 군을 '3개월까지 시작했는가'(exposed)가 아니라 '1년 안에 한 번이라도 시작했는가'(ever)로 나누면 위험비가 얼마로 나옵니까?

### 셀 21. 과제 1 정답. 랜드마크 뒤의 처방까지 넣어 군을 나누면

In [ ]:
L = 365 * 3 / 12
lm_w = d[d["fu_days"] > L].copy()          # 3개월 생존자
lm_w["t_lm"] = lm_w["fu_days"] - L
print(lm_w.groupby("ever")["died"].agg(["count", "sum"]))
m1 = CoxPHFitter().fit(lm_w, "t_lm", "died", formula="ever")
print("HR (95% CI):", hr_ci(m1, "ever"))

late = lm_w[lm_w["rx_day"] > L]            # 3개월 뒤에 시작한 사람
print(len(late), "명의 3개월부터 첫 처방까지:",
      round((late["rx_day"] - L).sum() / 365), "인년")

**과제 2.** 사용군은 첫 처방일부터, 비사용군은 퇴원일부터 추적하면(첫 처방 전 시간을 버리면) 사망률, 발생률비, 위험비가 어떻게 됩니까?

### 셀 22. 과제 2 정답. 첫 처방 전 시간을 버리기만 하면

In [ ]:
ex2 = pd.concat([
    pd.DataFrame({"t": u["fu_days"] - u["rx_day"],   # 첫 처방일부터
                  "died": u["died"], "ever": 1}),
    pd.DataFrame({"t": nu["fu_days"],                # 퇴원일부터
                  "died": nu["died"], "ever": 0})])
py2 = ex2.groupby("ever")["t"].sum() / 365
rate2 = ex2.groupby("ever")["died"].sum() / py2 * 100
print("인년:", py2.round(0).to_dict())
print("100인년당 사망률:", rate2.round(1).to_dict(),
      " 발생률비:", round(rate2[1] / rate2[0], 2))
m2 = CoxPHFitter().fit(ex2, "t", "died", formula="ever")
print("HR (95% CI):", hr_ci(m2, "ever"))

**과제 3.** 첫 처방 뒤 30일까지를 비노출로 세는 시차 30일의 긴 형식 자료를 만들고, 노출 상태별 사망 수와 인년, 시간의존 Cox 모형의 위험비를 구하세요.

### 셀 23. 과제 3 정답. 시차 30일을 둔 노출

In [ ]:
def make_long(d, lag=0):
    sw = d["rx_day"] + lag              # 노출로 바뀌는 날
    s = sw < d["fu_days"]               # 추적 중에 바뀌는 사람
    a, b = d[s], d[~s]
    before = pd.DataFrame({"id": a["id"], "start": 0.0,
                           "stop": sw[s], "drugx": 0, "died": 0})
    after = pd.DataFrame({"id": a["id"], "start": sw[s],
                          "stop": a["fu_days"], "drugx": 1,
                          "died": a["died"]})
    never = pd.DataFrame({"id": b["id"], "start": 0.0,
                          "stop": b["fu_days"], "drugx": 0,
                          "died": b["died"]})
    out = pd.concat([before, after, never])
    return out.sort_values(["id", "start"]).reset_index(drop=True)

long30 = make_long(d, lag=30)
long30["py"] = (long30["stop"] - long30["start"]) / 365
print(len(long30), "줄")
print(long30.groupby("drugx").agg(deaths=("died", "sum"),
                                  py=("py", "sum")).round(0))
m3 = CoxTimeVaryingFitter().fit(long30, id_col="id",
    event_col="died", start_col="start", stop_col="stop",
    formula="drugx")
print("시차 30일 HR (95% CI):", hr_ci(m3, "drugx"))

chk = add_covariate_to_timeline(base, rx, id_col="id",
    duration_col="rx_day", event_col="died", delay=30)
print("lifelines delay=30:", len(chk), "줄")